In [1]:
from run_single_experiment import run_experiment_1, run_experiment_2, run_experiment_3, run_experiment_4

/Users/ukun/anaconda3/envs/CTGAN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD = /Users/ukun/Desktop/captcha
captcha_data exists? True
secrets.yaml exists? True
sample type dirs: ['./captcha_data/Unusual_Detection', './captcha_data/Connect_icon', './captcha_data/Select_Animal', './captcha_data/Coordinates', './captcha_data/Geometry_Click']
{
  "providers": {
    "openai": {
      "api_key": "REDACTED_OPENAI_API_KEY"
    },
    "anthropic": {
      "api_key": "REDACTED_OPENAI_API_KEY"
    },
    "gemini": {
      "api_key": "REDACTED_GEMINI_API_KEY"
    },
    "fireworks": {
      "api_key": "fw_3ZH7VsD9TY6GCuUxUQ1uXn2W",
      "base_url": "https://api.fireworks.ai/inference/v1"
    }
  },
  "pricing": {
    "openai": {
      "gpt-5": {
        "in_per_1k": 0.00125,
        "out_per_1k": 0.01
      },
      "gpt-5-chat-latest": {
        "in_per_1k": 0.00125,
        "out_per_1k": 0.01
      },
      "gpt-5.1": {
        "in_per_1k": 0.00125,
        "out_per_1k": 0.01
      }
    },
    "anthropic": {
      "claude-opus-4-1": {
        "in_per_1k": 0.015,
   

In [2]:
from pathlib import Path

RESULTS_BASE = Path("./results")
ERROR_BASE = Path("./error_analysis")

def _norm_model_name(model: str) -> str:
    return model.replace("/", "_")

def exp_results_dir(exp_name: str, provider: str, model: str) -> str:
    path = RESULTS_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)

def exp_out_csv(exp_name: str, provider: str, model: str, filename: str = "results.csv") -> str:
    return str(Path(exp_results_dir(exp_name, provider, model)) / filename)

def exp_error_dir(exp_name: str, provider: str, model: str) -> str:
    path = ERROR_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


In [3]:
SUPPORTED_TYPES = {
    "Dice_Count",
    "Geometry_Click",
    "Image_Matching",
    "Patch_Select",
    "Place_Dot",

    "Bingo",
    "Click_Order",
    "Dart_Count",
    "Image_Recognition",
    "Misleading_Click",
    "Object_Match",
    "Path_Finder",
    "Pick_Area",
    "Select_Animal",
    "Unusual_Detection",
    "Coordinates",
    "Connect_Icon",
    "Rotation_Match",
}

ERROR_TYPES = {
    "Dice_Count",
    "Click_Order",
    "Patch_Select",
    "Place_Dot"
}

In [7]:
# 实验一：Ground Truth Prompts
provider = "anthropic"
model = "claude-sonnet-4-5"
reasoning_effort = None
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result1 = run_experiment_1(
    types=SUPPORTED_TYPES,
    max_per_type=20,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp1", provider, storage_model),
    thinking=False,
    thinking_options=thinking_opts,
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp1", provider, storage_model),
    enable_error_analysis=False,
    error_analysis_dir=exp_error_dir("exp1", provider, storage_model),
    collect_reasoning=False,
    timeout_sec=600
)



🔵 实验一：Ground Truth Prompts
Provider: anthropic
Model: claude-sonnet-4-5
Thinking: False

[INFO] 将评测 303 道题（types={'Path_Finder', 'Dart_Count', 'Click_Order', 'Unusual_Detection', 'Image_Recognition', 'Misleading_Click', 'Coordinates', 'Object_Match', 'Rotation_Match', 'Geometry_Click', 'Bingo', 'Place_Dot', 'Image_Matching', 'Dice_Count', 'Patch_Select', 'Connect_Icon', 'Select_Animal', 'Pick_Area'}）


Evaluating: 100% 303/303 [40:49<00:00,  8.09s/it]  


[SUMMARY by type]
- Bingo              n= 20  pass@1=0.150  avg_e2e_ms=4528.8  avg_ttft_ms=4528.8  tokens_in=44042 tokens_out=1140 cost_usd=0.149226
- Click_Order        n= 10  pass@1=0.200  avg_e2e_ms=6560.0  avg_ttft_ms=6560.0  tokens_in=11231 tokens_out=1129 cost_usd=0.050628
- Connect_Icon       n= 20  pass@1=0.350  avg_e2e_ms=9256.0  avg_ttft_ms=9256.0  tokens_in=98646 tokens_out=1040 cost_usd=0.311538
- Coordinates        n= 18  pass@1=0.056  avg_e2e_ms=12135.9  avg_ttft_ms=12135.9  tokens_in=130031 tokens_out=468 cost_usd=0.397113
- Dart_Count         n= 20  pass@1=0.050  avg_e2e_ms=26415.0  avg_ttft_ms=26415.0  tokens_in=279716 tokens_out=1040 cost_usd=0.854748
- Dice_Count         n= 11  pass@1=0.000  avg_e2e_ms=7120.1  avg_ttft_ms=7120.1  tokens_in=24163 tokens_out=572 cost_usd=0.081069
- Geometry_Click     n= 15  pass@1=0.600  avg_e2e_ms=4789.6  avg_ttft_ms=4789.6  tokens_in=14175 tokens_out=960 cost_usd=0.056925
- Image_Matching     n= 19  pass@1=0.158  avg_e2e_ms=14278.7 

In [9]:
# 实验二：Optimized Prompts
provider = "openai"
model = "gpt-5.1"
reasoning_effort = "medium"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result2 = run_experiment_2(
    types=SUPPORTED_TYPES,
    prompts_file="./prompts_optimized.yaml",
    max_per_type=20,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp2", provider, storage_model),
    thinking=False,
    thinking_options={"effort": reasoning_effort},
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp2", provider, storage_model),
    enable_error_analysis=False,
    error_analysis_dir=exp_error_dir("exp2", provider, storage_model),
    collect_reasoning=False,
    timeout_sec=600
)


🟢 实验二：Optimized Prompts
Provider: openai
Model: gpt-5.1
Prompts file: ./prompts_optimized.yaml
Thinking: False

[INFO] 将评测 303 道题（types={'Path_Finder', 'Dart_Count', 'Click_Order', 'Unusual_Detection', 'Image_Recognition', 'Misleading_Click', 'Coordinates', 'Object_Match', 'Rotation_Match', 'Geometry_Click', 'Bingo', 'Place_Dot', 'Image_Matching', 'Dice_Count', 'Patch_Select', 'Connect_Icon', 'Select_Animal', 'Pick_Area'}）


Evaluating: 100% 303/303 [1:43:30<00:00, 20.50s/it]  


[SUMMARY by type]
- Bingo              n= 20  pass@1=0.950  avg_e2e_ms=38189.2  avg_ttft_ms=38182.0  tokens_in=21060 tokens_out=35031 cost_usd=0.376635
- Click_Order        n= 10  pass@1=0.100  avg_e2e_ms=16378.9  avg_ttft_ms=16376.1  tokens_in=8480 tokens_out=5827 cost_usd=0.068870
- Connect_Icon       n= 20  pass@1=0.850  avg_e2e_ms=20555.8  avg_ttft_ms=20549.9  tokens_in=74469 tokens_out=18580 cost_usd=0.278886
- Coordinates        n= 18  pass@1=0.722  avg_e2e_ms=32900.6  avg_ttft_ms=32892.4  tokens_in=97202 tokens_out=32073 cost_usd=0.442233
- Dart_Count         n= 20  pass@1=0.950  avg_e2e_ms=42049.4  avg_ttft_ms=42039.0  tokens_in=180490 tokens_out=16526 cost_usd=0.390873
- Dice_Count         n= 11  pass@1=0.000  avg_e2e_ms=42885.8  avg_ttft_ms=42875.1  tokens_in=13047 tokens_out=30607 cost_usd=0.322379
- Geometry_Click     n= 15  pass@1=0.467  avg_e2e_ms=8324.1  avg_ttft_ms=8322.4  tokens_in=7379 tokens_out=3266 cost_usd=0.041884
- Image_Matching     n= 19  pass@1=0.737  avg_e2

In [11]:
# 实验三：Until Correct
provider = "openai"
model = "gpt-5.1"
reasoning_effort = "none"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result3 = run_experiment_3(
    types=SUPPORTED_TYPES,
    prompts_file="./prompts_optimized.yaml",
    prompt_mode="auto",
    max_attempts_per_type=10,
    max_pool_per_type=20,
    use_full_dataset_pool=False,
    out_csv=exp_out_csv("exp3", provider, model),
    provider=provider,
    model=model,
    thinking=False,
    thinking_options=None,
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp3", provider, model),
    collect_reasoning=False
)



🟡 实验三：Until Correct Strategy
Provider: openai
Model: gpt-5.1
Max attempts per type: 10
Prompts file: ./prompts_optimized.yaml
Use full dataset pool: False


========== [Path_Finder] 开始评测（最多 10 次尝试） ==========
[Path_Finder] Attempt 1/10 • PID=path10.JPG
 ↳ OK  e2e=5653.1ms  cum=5653.1ms
[Path_Finder] 🎯 首次命中：PID=path10.JPG  用时累计=5653.1ms  尝试次数=1
[Path_Finder] 总结：attempts=1  cum_ms=5653.1  success=1  first_pid=path10.JPG

========== [Dart_Count] 开始评测（最多 10 次尝试） ==========
[Dart_Count] Attempt 1/10 • PID=dart_puzzle_2.json
 ↳ FAIL  e2e=19967.7ms  cum=19967.7ms
[Dart_Count] Attempt 2/10 • PID=dart_puzzle_18.json
 ↳ FAIL  e2e=26235.5ms  cum=46203.2ms
[Dart_Count] Attempt 3/10 • PID=dart_puzzle_20.json
 ↳ FAIL  e2e=24113.0ms  cum=70316.2ms
[Dart_Count] Attempt 4/10 • PID=dart_puzzle_5.json
 ↳ FAIL  e2e=22492.2ms  cum=92808.4ms
[Dart_Count] Attempt 5/10 • PID=dart_puzzle_19.json
 ↳ FAIL  e2e=28678.9ms  cum=121487.4ms
[Dart_Count] Attempt 6/10 • PID=dart_puzzle_5.json
 ↳ FAIL  e2e=16355.4ms  c

In [ ]:
# 实验四：Few-shot
provider = "openai"
model = "gpt-5"
reasoning_effort = "none"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result4 = run_experiment_4(
    prompts_file="./prompts_optimized.yaml",
    few_shot_file="./few_shot_examples.yaml",
    types=ERROR_TYPES,
    max_per_type=10,
    provider=provider,
    model=model,
    n_shot=2,
    thinking=False,
    thinking_options={"effort": reasoning_effort},
    enable_error_analysis=False,
    error_analysis_dir=exp_error_dir("exp4", provider, model),
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp4", provider, model),
    collect_reasoning=False,
    out_csv=exp_out_csv("exp4", provider, model),
    timeout_sec=600
)



🟣 实验四：Optimized Prompts + Few-shot Learning
Provider: openai
Model: gpt-5
Prompts file: ./prompts_optimized.yaml
Few-shot file: ./few_shot_examples.yaml
N-shot: 2
Note: 每种类型使用前 2 个样本作为示例

[INFO] Few-shot learning enabled
[INFO] Excluding 8 few-shot examples from testing
[INFO] 将评测 32 道题（types={'Patch_Select', 'Place_Dot', 'Dice_Count', 'Click_Order'}）


Evaluating: 100% 32/32 [33:20<00:00, 62.53s/it] 


[SUMMARY by type]
- Click_Order        n=  8  pass@1=0.250  avg_e2e_ms=61086.7  avg_ttft_ms=61082.7  tokens_in=21168 tokens_out=13592 cost_usd=0.162380
- Dice_Count         n=  8  pass@1=0.000  avg_e2e_ms=129185.3  avg_ttft_ms=129172.4  tokens_in=24104 tokens_out=38656 cost_usd=0.416690
- Patch_Select       n=  8  pass@1=0.000  avg_e2e_ms=18608.5  avg_ttft_ms=18606.6  tokens_in=14325 tokens_out=4295 cost_usd=0.060856
- Place_Dot          n=  8  pass@1=0.000  avg_e2e_ms=41217.2  avg_ttft_ms=41213.4  tokens_in=18008 tokens_out=11128 cost_usd=0.133790
[INFO] Token summary written to results/exp4/openai/gpt-5/exp4_fewshot_openai_gpt-5_token_summary.json

[OVERALL] tasks=32  pass@1=0.062  sum_e2e_ms=2000782.3  wall_ms=2000854.0
[OVERALL TOKENS] tokens_in=77605 tokens_out=67671 cost_usd=0.773716
[DONE] Pass@1 = 2/32 = 0.062 ; errors=0. 结果已保存到 results/exp4/openai/gpt-5/results.csv

✅ 实验四完成！
   结果文件: results/exp4/openai/gpt-5/results.csv
   注意：每种类型使用了 2 个样本作为示例，剩余样本用于测试
